## Movie Review Sentiment Analysis Using RNN

In [1]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv('train.tsv', sep='\t')
train.sample(10)

,PhraseId,SentenceId,Phrase,Sentiment
54050,54051,2685,suits the script,3
126330,126331,6791,is very choppy and monosyllabic despite the fa...,0
153481,153482,8383,reasonably intelligent,3
13916,13917,599,pile up,2
32246,32247,1511,as the remarkable ensemble cast brings them to...,3
78611,78612,4045,start and,2
81424,81425,4198,Dog Soldiers,2
120881,120882,6468,amusements,3
143627,143628,7795,lift -LRB- this -RRB- thrill-kill cat-and-mous...,2
13942,13943,600,they engage and even touch us,3


In [4]:
train.shape

(156060, 4)

In [127]:
train.loc[126330,['Phrase']]

,126330
Phrase,is very choppy and monosyllabic despite the fa...


In [19]:
train['Sentiment'].value_counts()

,count
Sentiment,
2,79582
3,32927
1,27273
4,9206
0,7072


In [5]:
X = train.loc[:,'Phrase']
y = train.loc[:,'Sentiment']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
X_train.shape,X_test.shape

((124848,), (31212,))

In [8]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.layers import Dense,SimpleRNN,Embedding,Flatten,Bidirectional,LSTM

tokenizer = Tokenizer(oov_token='<nothing>')

In [9]:
tokenizer.fit_on_texts(X_train)

In [28]:
len(tokenizer.word_counts)

15276

In [ ]:
tokenizer.word_index

In [10]:
tokenizer.document_count

124848

In [ ]:
train_seq = tokenizer.texts_to_sequences(X_train)
train_seq

In [ ]:
test_seq = tokenizer.texts_to_sequences(X_test)
test_seq

In [13]:
train_seq = pad_sequences(train_seq,padding='post')
train_seq,len(train_seq[0])

(array([[1677,    0,    0, ...,    0,    0,    0],
        [2173,   45,    0, ...,    0,    0,    0],
        [  71,  285,   34, ...,    0,    0,    0],
        ...,
        [   3,  299,  215, ...,    0,    0,    0],
        [  53,   23,   83, ...,    0,    0,    0],
        [   5, 1083,  127, ...,    0,    0,    0]], dtype=int32),
 49)

In [14]:
test_seq = pad_sequences(test_seq,padding='post',maxlen=len(train_seq[0]))
test_seq,len(test_seq[0])

(array([[   7,   12, 3799, ...,    0,    0,    0],
        [ 222,   61,    5, ...,    0,    0,    0],
        [   4,  174,  533, ...,    0,    0,    0],
        ...,
        [   2,  751, 4126, ...,    0,    0,    0],
        [2141,  387, 5542, ...,    0,    0,    0],
        [   2,  470, 3133, ...,    0,    0,    0]], dtype=int32),
 49)

In [110]:
model = Sequential()

model.add(Embedding(15277,2))
# model.add(SimpleRNN(32,input_shape=(train_seq.shape[1],1),return_sequences=False))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(5,activation='sigmoid'))

model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [111]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [23]:
train_seq.shape,y_train.shape

((124848, 49), (124848,))

In [112]:
# Removed reshape operations. The Embedding layer expects 2D input.
# train_seq_reshaped = train_seq.reshape(train_seq.shape[0], train_seq.shape[1], 1)
# test_seq_reshaped = test_seq.reshape(test_seq.shape[0], test_seq.shape[1], 1)

# One-hot encode y_train and y_test
y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=5)
y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=5)

model.fit(
    train_seq,
    y_train_one_hot,
    epochs=10,
    validation_data=(test_seq, y_test_one_hot)
)

Epoch 1/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 48s 11ms/step - accuracy: 0.5532 - loss: 1.1178 - val_accuracy: 0.5883 - val_loss: 1.0350
Epoch 2/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 43s 11ms/step - accuracy: 0.6363 - loss: 0.9051 - val_accuracy: 0.6468 - val_loss: 0.8681
Epoch 3/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 41s 11ms/step - accuracy: 0.6756 - loss: 0.7929 - val_accuracy: 0.6565 - val_loss: 0.8423
Epoch 4/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 41s 10ms/step - accuracy: 0.6883 - loss: 0.7566 - val_accuracy: 0.6617 - val_loss: 0.8370
Epoch 5/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 47s 12ms/step - accuracy: 0.6952 - loss: 0.7356 - val_accuracy: 0.6625 - val_loss: 0.8314
Epoch 6/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 41s 11ms/step - accuracy: 0.7026 - loss: 0.7196 - val_accuracy: 0.6664 - val_loss: 0.8320
Epoch 7/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 42s 11ms/step - accuracy: 0.7065 - loss: 0.7074 - val_accuracy: 0.6644 - val_loss: 0.8349
Epoch 8/10
3902/3902 ━━━━━━━━━━━━━━━━━━━━ 41s 11ms/step - accuracy: 0.7107 -

In [134]:
import numpy as np

txt_sentence = "hate it"

# Tokenize the sentence, treating it as a single text entry (wrap in a list)
txt_sequence = tokenizer.texts_to_sequences([txt_sentence])

# Pad the sequence to the same length as training data
txt_padded = pad_sequences(txt_sequence, padding='post', maxlen=train_seq.shape[1])

# Make prediction
prediction_probabilities = model.predict(txt_padded)

print("Prediction Probabilities for each sentiment class:", prediction_probabilities)

# To get the predicted sentiment (class with highest probability):
predicted_sentiment = np.argmax(prediction_probabilities, axis=1)
print("Predicted Sentiment (class index):", predicted_sentiment[0])

# Note: Sentiment classes are 0: Very Negative, 1: Negative, 2: Neutral, 3: Positive, 4: Very Positive

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Prediction Probabilities for each sentiment class: [[0.5498572  0.9440498  0.8646867  0.3449097  0.03114651]]
Predicted Sentiment (class index): 1
